# Equal-Weight S&P 500 Index Fund

## Introduction & Library Imports

The S&P 500 is the world's most popular stock market index. The largest fund that is benchmarked to this index is the SPDR® S&P 500® ETF Trust. It has more than US$250 billion of assets under management.

The goal of this section of the course is to create a Python script that will accept the value of your portfolio and tell you how many shares of each S&P 500 constituent you should purchase to get an equal-weight version of the index fund.

## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [19]:
%pip install yfinance

  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached websockets-16.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (6.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cffi-2.0.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 2.7 MB/s  0:00:01 eta 0:00:01


In [20]:
import numpy as np
import pandas as pd
import requests
import xlsxwriter
import math
import yfinance as yf

## Importing Our List of Stocks

The next thing we need to do is import the constituents of the S&P 500.

These constituents change over time, so in an ideal world you would connect directly to the index provider (Standard & Poor's) and pull their real-time constituents on a regular basis.

Paying for access to the index provider's API is outside of the scope of this course. 

There's a static version of the S&P 500 constituents available here. [Click this link to download them now](https://drive.google.com/file/d/1ZJSpbY69DVckVZlO9cC6KkgfSufybcHN/view?usp=sharing). Move this file into the `starter-files` folder so it can be accessed by other files in that directory.

Now it's time to import these stocks to our Jupyter Notebook file.

In [21]:
stocks = pd.read_csv('sp_500_stocks.csv')
stocks

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
...,...
500,YUM
501,ZBH
502,ZBRA
503,ZION


## Acquiring an API Token

Now it's time to import our IEX Cloud API token. This is the data provider that we will be using throughout this course.

API tokens (and other sensitive information) should be stored in a `secrets.py` file that doesn't get pushed to your local Git repository. We'll be using a sandbox API token in this course, which means that the data we'll use is randomly-generated and (more importantly) has no cost associated with it.

[Click here](http://nickmccullum.com/algorithmic-trading-python/secrets.py) to download your `secrets.py` file. Move the file into the same directory as this Jupyter Notebook before proceeding.

In [22]:
from APIKEY import IEX_CLOUD_API_TOKEN

## Making Our First API Call

Now it's time to structure our API calls to IEX cloud. 

We need the following information from the API:

* Market capitalization for each stock
* Price of each stock



In [52]:
symbol = 'AAPL'

ticker = yf.Ticker(symbol)
data = ticker.info

# # Common fields you'd get from IEX /quote endpoint
# print(data['currentPrice'])        # latestPrice
# print(data['marketCap'])           # marketCap
# print(data['trailingPE'])          # peRatio
# print(data['fiftyTwoWeekHigh'])    # week52High
# print(data['fiftyTwoWeekLow'])     # week52Low

print(data)

{'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applications and

## Parsing Our API Call

The API call that we executed in the last code block contains all of the information required to build our equal-weight S&P 500 strategy. 

With that said, the data isn't in a proper format yet. We need to parse it first.

In [ ]:
price = data['currentPrice']
market_cap = data['marketCap']

3805292789760


## Adding Our Stocks Data to a Pandas DataFrame

The next thing we need to do is add our stock's price and market capitalization to a pandas DataFrame. Think of a DataFrame like the Python version of a spreadsheet. It stores tabular data.

In [36]:
my_columns = [ 'Ticker', 'Stock Price', 'Market Capitalization', 'Number of Shares to Buy']
final_dataframe = pd.DataFrame(columns = my_columns)
final_dataframe

,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy


In [49]:
final_dataframe = pd.concat([
    final_dataframe,
    pd.Series(
    [
        symbol,
        price,
        market_cap,
        'N/A'
    ],
    index = my_columns
    ),
], ignore_index=True
)

print(final_dataframe)

   Ticker Stock Price Market Capitalization Number of Shares to Buy  \
0    AAPL       258.9         3805292789760                     N/A   
1    AAPL       258.9         3805292789760                     N/A   
2     NaN         NaN                   NaN                     NaN   
3     NaN         NaN                   NaN                     NaN   
4     NaN         NaN                   NaN                     NaN   
5     NaN         NaN                   NaN                     NaN   
6     NaN         NaN                   NaN                     NaN   
7     NaN         NaN                   NaN                     NaN   
8     NaN         NaN                   NaN                     NaN   
9     NaN         NaN                   NaN                     NaN   
10    NaN         NaN                   NaN                     NaN   
11    NaN         NaN                   NaN                     NaN   
12    NaN         NaN                   NaN                     NaN   
13    

## Looping Through The Tickers in Our List of Stocks

Using the same logic that we outlined above, we can pull data for all S&P 500 stocks and store their data in the DataFrame using a `for` loop.

In [ ]:
for symbol in stocks['Ticker']:
    print(pd.DataFrame([symbol], columns=['Ticker']))

  Ticker
0      A
  Ticker
0    AAL
  Ticker
0    AAP
  Ticker
0   AAPL
  Ticker
0   ABBV
  Ticker
0    ABC
  Ticker
0   ABMD
  Ticker
0    ABT
  Ticker
0    ACN
  Ticker
0   ADBE
  Ticker
0    ADI
  Ticker
0    ADM
  Ticker
0    ADP
  Ticker
0   ADSK
  Ticker
0    AEE
  Ticker
0    AEP
  Ticker
0    AES
  Ticker
0    AFL
  Ticker
0    AIG
  Ticker
0    AIV
  Ticker
0    AIZ
  Ticker
0    AJG
  Ticker
0   AKAM
  Ticker
0    ALB
  Ticker
0   ALGN
  Ticker
0    ALK
  Ticker
0    ALL
  Ticker
0   ALLE
  Ticker
0   ALXN
  Ticker
0   AMAT
  Ticker
0   AMCR
  Ticker
0    AMD
  Ticker
0    AME
  Ticker
0   AMGN
  Ticker
0    AMP
  Ticker
0    AMT
  Ticker
0   AMZN
  Ticker
0   ANET
  Ticker
0   ANSS
  Ticker
0   ANTM
  Ticker
0    AON
  Ticker
0    AOS
  Ticker
0    APA
  Ticker
0    APD
  Ticker
0    APH
  Ticker
0   APTV
  Ticker
0    ARE
  Ticker
0    ATO
  Ticker
0   ATVI
  Ticker
0    AVB
  Ticker
0   AVGO
  Ticker
0    AVY
  Ticker
0    AWK
  Ticker
0    AXP
  Ticker
0    AZO
  Ticker
0

In [58]:
final_dataframe = pd.DataFrame(columns=my_columns)
rows = []

for stock in stocks['Ticker']:
    ticker = yf.Ticker(stock)
    data = ticker.info
    
    rows.append([
        stock,
        data.get('currentPrice', 'N/A'),
        data.get('marketCap', 'N/A'),
        'N/A'
    ])
    
    print(f"Fetched: {stock}")

final_dataframe = pd.DataFrame(rows, columns=my_columns)
final_dataframe

Fetched: A
Fetched: AAL
Fetched: AAP
Fetched: AAPL
Fetched: ABBV
Fetched: ABC
Fetched: ABMD
Fetched: ABT
Fetched: ACN
Fetched: ADBE
Fetched: ADI
Fetched: ADM
Fetched: ADP
Fetched: ADSK
Fetched: AEE
Fetched: AEP
Fetched: AES
Fetched: AFL
Fetched: AIG
Fetched: AIV
Fetched: AIZ
Fetched: AJG
Fetched: AKAM
Fetched: ALB
Fetched: ALGN
Fetched: ALK
Fetched: ALL
Fetched: ALLE
Fetched: ALXN
Fetched: AMAT
Fetched: AMCR
Fetched: AMD
Fetched: AME
Fetched: AMGN
Fetched: AMP
Fetched: AMT
Fetched: AMZN
Fetched: ANET


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}


Fetched: ANSS
Fetched: ANTM
Fetched: AON
Fetched: AOS
Fetched: APA
Fetched: APD
Fetched: APH
Fetched: APTV
Fetched: ARE
Fetched: ATO
Fetched: ATVI
Fetched: AVB
Fetched: AVGO
Fetched: AVY
Fetched: AWK
Fetched: AXP
Fetched: AZO
Fetched: BA
Fetched: BAC
Fetched: BAX
Fetched: BBY
Fetched: BDX
Fetched: BEN
Fetched: BF.B
Fetched: BIIB
Fetched: BIO
Fetched: BK
Fetched: BKNG
Fetched: BKR
Fetched: BLK
Fetched: BLL
Fetched: BMY
Fetched: BR
Fetched: BRK.B
Fetched: BSX
Fetched: BWA
Fetched: BXP
Fetched: C
Fetched: CAG
Fetched: CAH
Fetched: CARR
Fetched: CAT
Fetched: CB
Fetched: CBOE
Fetched: CBRE
Fetched: CCI
Fetched: CCL
Fetched: CDNS
Fetched: CDW
Fetched: CE
Fetched: CERN
Fetched: CF
Fetched: CFG
Fetched: CHD
Fetched: CHRW
Fetched: CHTR
Fetched: CI
Fetched: CINF
Fetched: CL
Fetched: CLX


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}


Fetched: CMA
Fetched: CMCSA
Fetched: CME
Fetched: CMG
Fetched: CMI
Fetched: CMS
Fetched: CNC
Fetched: CNP
Fetched: COF
Fetched: COG
Fetched: COO
Fetched: COP
Fetched: COST
Fetched: COTY
Fetched: CPB
Fetched: CPRT
Fetched: CRM
Fetched: CSCO
Fetched: CSX
Fetched: CTAS
Fetched: CTL
Fetched: CTSH
Fetched: CTVA
Fetched: CTXS
Fetched: CVS
Fetched: CVX
Fetched: CXO
Fetched: D
Fetched: DAL
Fetched: DD
Fetched: DE
Fetched: DFS
Fetched: DG
Fetched: DGX
Fetched: DHI
Fetched: DHR
Fetched: DIS
Fetched: DISCA
Fetched: DISCK
Fetched: DISH
Fetched: DLR
Fetched: DLTR
Fetched: DOV
Fetched: DOW
Fetched: DPZ
Fetched: DRE
Fetched: DRI
Fetched: DTE
Fetched: DUK
Fetched: DVA
Fetched: DVN
Fetched: DXC
Fetched: DXCM
Fetched: EA
Fetched: EBAY
Fetched: ECL
Fetched: ED
Fetched: EFX
Fetched: EIX
Fetched: EL
Fetched: EMN
Fetched: EMR
Fetched: EOG
Fetched: EQIX
Fetched: EQR
Fetched: ES
Fetched: ESS
Fetched: ETFC
Fetched: ETN
Fetched: ETR
Fetched: EVRG
Fetched: EW
Fetched: EXC
Fetched: EXPD
Fetched: EXPE
Fetched: EXR

...

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}


Fetched: K
Fetched: KEY
Fetched: KEYS
Fetched: KHC
Fetched: KIM
Fetched: KLAC
Fetched: KMB
Fetched: KMI
Fetched: KMX
Fetched: KO
Fetched: KR
Fetched: KSS
Fetched: KSU
Fetched: L
Fetched: LB
Fetched: LDOS
Fetched: LEG
Fetched: LEN
Fetched: LH
Fetched: LHX
Fetched: LIN
Fetched: LKQ
Fetched: LLY
Fetched: LMT
Fetched: LNC
Fetched: LNT
Fetched: LOW
Fetched: LRCX
Fetched: LUV
Fetched: LVS
Fetched: LW
Fetched: LYB
Fetched: LYV
Fetched: MA
Fetched: MAA
Fetched: MAR
Fetched: MAS
Fetched: MCD
Fetched: MCHP
Fetched: MCK
Fetched: MCO
Fetched: MDLZ
Fetched: MDT
Fetched: MET
Fetched: MGM
Fetched: MHK
Fetched: MKC
Fetched: MKTX
Fetched: MLM
Fetched: MMC
Fetched: MMM
Fetched: MNST
Fetched: MO
Fetched: MOS
Fetched: MPC
Fetched: MRK
Fetched: MRO
Fetched: MS
Fetched: MSCI
Fetched: MSFT
Fetched: MSI
Fetched: MTB
Fetched: MTD
Fetched: MU
Fetched: MXIM
Fetched: MYL
Fetched: NBL
Fetched: NCLH
Fetched: NDAQ
Fetched: NEE
Fetched: NEM
Fetched: NFLX
Fetched: NI
Fetched: NKE
Fetched: NLOK
Fetched: NLSN
Fetched: N

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


Fetched: WBA
Fetched: WDC
Fetched: WEC
Fetched: WELL
Fetched: WFC
Fetched: WHR
Fetched: WLTW
Fetched: WM
Fetched: WMB
Fetched: WMT
Fetched: WRB
Fetched: WRK
Fetched: WST
Fetched: WU
Fetched: WY
Fetched: WYNN
Fetched: XEL
Fetched: XLNX
Fetched: XOM
Fetched: XRAY
Fetched: XRX
Fetched: XYL
Fetched: YUM
Fetched: ZBH
Fetched: ZBRA
Fetched: ZION
Fetched: ZTS


,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,115.02,32532215808,N/A
1,AAL,11.185,7385506816,N/A
2,AAP,54.54,3290354944,N/A
3,AAPL,257.5313,3785524248576,N/A
4,ABBV,210.77,372802060288,N/A
...,...,...,...,...
500,YUM,161.03,44513546240,N/A
501,ZBH,92.72,18140854272,N/A
502,ZBRA,220.36,10839883776,N/A
503,ZION,60.96,9015303168,N/A


## Using Batch API Calls to Improve Performance

Batch API calls are one of the easiest ways to improve the performance of your code.

This is because HTTP requests are typically one of the slowest components of a script.

Also, API providers will often give you discounted rates for using batch API calls since they are easier for the API provider to respond to.

IEX Cloud limits their batch API calls to 100 tickers per request. Still, this reduces the number of API calls we'll make in this section from 500 to 5 - huge improvement! In this section, we'll split our list of stocks into groups of 100 and then make a batch API call for each group.

In [59]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst"""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [68]:
symbol_groups = list(chunks(stocks['Ticker'], 100))
rows = []
final_dataframe = pd.DataFrame(columns=my_columns)

for symbol_group in symbol_groups:
    # yfinance batch download - much faster than one by one
    tickers = yf.Tickers(' '.join(symbol_group))
    
    for symbol in symbol_group:
        try:
            data = tickers.tickers[symbol].info
            rows.append([
                symbol,
                data.get('currentPrice', 'N/A'),
                data.get('marketCap', 'N/A'),
                'N/A'
            ])
            print(f"Fetched: {symbol}")
        except Exception as e:
            print(f"Skipping {symbol}: {e}")

final_dataframe = pd.DataFrame(rows, columns=my_columns)
final_dataframe

Fetched: A
Fetched: AAL
Fetched: AAP
Fetched: AAPL
Fetched: ABBV
Fetched: ABC
Fetched: ABMD
Fetched: ABT
Fetched: ACN
Fetched: ADBE
Fetched: ADI
Fetched: ADM
Fetched: ADP
Fetched: ADSK
Fetched: AEE
Fetched: AEP
Fetched: AES
Fetched: AFL
Fetched: AIG
Fetched: AIV
Fetched: AIZ
Fetched: AJG
Fetched: AKAM
Fetched: ALB
Fetched: ALGN
Fetched: ALK
Fetched: ALL
Fetched: ALLE
Fetched: ALXN
Fetched: AMAT
Fetched: AMCR
Fetched: AMD
Fetched: AME
Fetched: AMGN
Fetched: AMP
Fetched: AMT
Fetched: AMZN
Fetched: ANET


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}


Fetched: ANSS
Fetched: ANTM
Fetched: AON
Fetched: AOS
Fetched: APA
Fetched: APD
Fetched: APH
Fetched: APTV
Fetched: ARE
Fetched: ATO
Fetched: ATVI
Fetched: AVB
Fetched: AVGO
Fetched: AVY
Fetched: AWK
Fetched: AXP
Fetched: AZO
Fetched: BA
Fetched: BAC
Fetched: BAX
Fetched: BBY
Fetched: BDX
Fetched: BEN
Fetched: BF.B
Fetched: BIIB
Fetched: BIO
Fetched: BK
Fetched: BKNG
Fetched: BKR
Fetched: BLK
Fetched: BLL
Fetched: BMY
Fetched: BR
Fetched: BRK.B
Fetched: BSX
Fetched: BWA
Fetched: BXP
Fetched: C
Fetched: CAG
Fetched: CAH
Fetched: CARR
Fetched: CAT
Fetched: CB
Fetched: CBOE
Fetched: CBRE
Fetched: CCI
Fetched: CCL
Fetched: CDNS
Fetched: CDW
Fetched: CE
Fetched: CERN
Fetched: CF
Fetched: CFG
Fetched: CHD
Fetched: CHRW
Fetched: CHTR
Fetched: CI
Fetched: CINF
Fetched: CL
Fetched: CLX


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}


Fetched: CMA
Fetched: CMCSA
Fetched: CME
Fetched: CMG
Fetched: CMI
Fetched: CMS
Fetched: CNC
Fetched: CNP
Fetched: COF
Fetched: COG
Fetched: COO
Fetched: COP
Fetched: COST
Fetched: COTY
Fetched: CPB
Fetched: CPRT
Fetched: CRM
Fetched: CSCO
Fetched: CSX
Fetched: CTAS
Fetched: CTL
Fetched: CTSH
Fetched: CTVA
Fetched: CTXS
Fetched: CVS
Fetched: CVX
Fetched: CXO
Fetched: D
Fetched: DAL
Fetched: DD
Fetched: DE
Fetched: DFS
Fetched: DG
Fetched: DGX
Fetched: DHI
Fetched: DHR
Fetched: DIS
Fetched: DISCA
Fetched: DISCK
Fetched: DISH
Fetched: DLR
Fetched: DLTR
Fetched: DOV
Fetched: DOW
Fetched: DPZ
Fetched: DRE
Fetched: DRI
Fetched: DTE
Fetched: DUK
Fetched: DVA
Fetched: DVN
Fetched: DXC
Fetched: DXCM
Fetched: EA
Fetched: EBAY
Fetched: ECL
Fetched: ED
Fetched: EFX
Fetched: EIX
Fetched: EL
Fetched: EMN
Fetched: EMR
Fetched: EOG
Fetched: EQIX
Fetched: EQR
Fetched: ES
Fetched: ESS
Fetched: ETFC
Fetched: ETN
Fetched: ETR
Fetched: EVRG
Fetched: EW
Fetched: EXC
Fetched: EXPD
Fetched: EXPE
Fetched: EXR

...

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}


Fetched: K
Fetched: KEY
Fetched: KEYS
Fetched: KHC
Fetched: KIM
Fetched: KLAC
Fetched: KMB
Fetched: KMI
Fetched: KMX
Fetched: KO
Fetched: KR
Fetched: KSS
Fetched: KSU
Fetched: L
Fetched: LB
Fetched: LDOS
Fetched: LEG
Fetched: LEN
Fetched: LH
Fetched: LHX
Fetched: LIN
Fetched: LKQ
Fetched: LLY
Fetched: LMT
Fetched: LNC
Fetched: LNT
Fetched: LOW
Fetched: LRCX
Fetched: LUV
Fetched: LVS
Fetched: LW
Fetched: LYB
Fetched: LYV
Fetched: MA
Fetched: MAA
Fetched: MAR
Fetched: MAS
Fetched: MCD
Fetched: MCHP
Fetched: MCK
Fetched: MCO
Fetched: MDLZ
Fetched: MDT
Fetched: MET
Fetched: MGM
Fetched: MHK
Fetched: MKC
Fetched: MKTX
Fetched: MLM
Fetched: MMC
Fetched: MMM
Fetched: MNST
Fetched: MO
Fetched: MOS
Fetched: MPC
Fetched: MRK
Fetched: MRO
Fetched: MS
Fetched: MSCI
Fetched: MSFT
Fetched: MSI
Fetched: MTB
Fetched: MTD
Fetched: MU
Fetched: MXIM
Fetched: MYL
Fetched: NBL
Fetched: NCLH
Fetched: NDAQ
Fetched: NEE
Fetched: NEM
Fetched: NFLX
Fetched: NI
Fetched: NKE
Fetched: NLOK
Fetched: NLSN
Fetched: N

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


Fetched: WBA
Fetched: WDC
Fetched: WEC
Fetched: WELL
Fetched: WFC
Fetched: WHR
Fetched: WLTW
Fetched: WM
Fetched: WMB
Fetched: WMT
Fetched: WRB
Fetched: WRK
Fetched: WST
Fetched: WU
Fetched: WY
Fetched: WYNN
Fetched: XEL
Fetched: XLNX
Fetched: XOM
Fetched: XRAY
Fetched: XRX
Fetched: XYL
Fetched: YUM
Fetched: ZBH
Fetched: ZBRA
Fetched: ZION
Fetched: ZTS


,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,114.65,32427565056,N/A
1,AAL,11.1696,7375337472,N/A
2,AAP,54.34,3278289152,N/A
3,AAPL,257.515,3784936521728,N/A
4,ABBV,210.73,372731281408,N/A
...,...,...,...,...
500,YUM,160.67,44414029824,N/A
501,ZBH,92.845,18165309440,N/A
502,ZBRA,219.76,10810369024,N/A
503,ZION,61.05,9028613120,N/A


## Calculating the Number of Shares to Buy

As you can see in the DataFrame above, we stil haven't calculated the number of shares of each stock to buy.

We'll do that next.

In [84]:
portfolio_size = input('Enter the value of your portfolio: ')

try:
    val = float(portfolio_size)
except ValueError:
    print("That is not a number! \nPlease try again")
    portfolio_size = input('Enter the value of your portfolio: ')
    val = float(portfolio_size)

In [85]:
# Reset the column as float to remove the 'N/A' string dtype
final_dataframe['Number of Shares to Buy'] = 0.0

position_size = val / len(final_dataframe.index)
for i in range(0, len(final_dataframe.index)):
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe.loc[i, 'Stock Price'])

final_dataframe

,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,114.6500,3.242757e+10,194.0
1,AAL,11.1696,7.375337e+09,1993.0
2,AAP,54.3400,3.278289e+09,409.0
3,AAPL,257.5150,3.784937e+12,86.0
4,ABBV,210.7300,3.727313e+11,105.0
...,...,...,...,...
444,YUM,160.6700,4.441403e+10,138.0
445,ZBH,92.8450,1.816531e+10,239.0
446,ZBRA,219.7600,1.081037e+10,101.0
447,ZION,61.0500,9.028613e+09,364.0


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

### Initializing our XlsxWriter Object

In [116]:
writer = pd.ExcelWriter('recomm trades.xlsx', engine='xlsxwriter')
final_dataframe.to_excel(writer, sheet_name='Recommended Trades', index=False)
writer.close()

### Creating the Formats We'll Need For Our `.xlsx` File

Formats include colors, fonts, and also symbols like `%` and `$`. We'll need four main formats for our Excel document:
* String format for tickers
* \\$XX.XX format for stock prices
* \\$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

In [117]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_format = writer.book.add_format(
    {
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)
dollar_format = writer.book.add_format(
    {
        'num_format': '$0.00',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)
integer_format = writer.book.add_format(
    {
        'num_format': '0',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)

### Applying the Formats to the Columns of Our `.xlsx` File

We can use the `set_column` method applied to the `writer.sheets['Recommended Trades']` object to apply formats to specific columns of our spreadsheets.

Here's an example:

```python
writer.sheets['Recommended Trades'].set_column('B:B', #This tells the method to apply the format to column B
                     18, #This tells the method to apply a column width of 18 pixels
                     string_template #This applies the format 'string_template' to the column
                    )
```

In [ ]:
# writer.sheets['Recommended Trades'].set_column('A:A', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('B:B', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('C:C', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('D:D', 18, string_format)
# writer.close()

# writer.sheets['Recommended Trades'].write('A1', 'Ticker', string_format)
# writer.sheets['Recommended Trades'].write('B1', 'Stock Price', dollar_format)
# writer.sheets['Recommended Trades'].write('C1', 'Market Capitalization', string_format)
# writer.sheets['Recommended Trades'].write('D1', 'Number of Shares to Buy', integer_format)

0

This code works, but it violates the software principle of "Don't Repeat Yourself". 

Let's simplify this by putting it in 2 loops:

In [118]:
column_formats = {
    'A': ['Ticker', string_format],
    'B': ['Stock Price', dollar_format],
    'C': ['Market Capitalisation', dollar_format],
    'D': ['Number of Shares to Buy', integer_format]
}

for column in column_formats.keys():
    writer.sheets['Recommended Trades'].write(f'{column}1', column_formats[column][0], column_formats[column][1])
    writer.sheets['Recommended Trades'].set_column(f'{column}:{column}', 18, column_formats[column][1])

writer.close()

## Saving Our Excel Output

Saving our Excel file is very easy:

In [ ]:
writer.close()